# Notebook 05 — Stage 1B: best-source analysis fields

**Role in the pipeline:** picks the *best available* value per row for
each chemistry variable (TA, observed pH, calculated pH, pCO₂, DIC),
records *which source* each value came from, classifies sample / CRM /
standard rows, and produces the analysis-ready subset that Stage 2 reads.

```text
04_stage1a.ipynb
   └── <stage1a_out>/data/staged.csv
                       │
                       ▼
          05_stage1b.ipynb            ◄── THIS NOTEBOOK
            • coalesce TA/pH/pCO2/DIC across precedence lists
            • record per-row provenance (which column won)
            • normalise QC status, TA units, pH scale
            • classify is_sample_row
            • two analysis gates: safe_for_analysis_qc / _strict
            • write analysis_fields.csv (everything) +
              analysis_ready_samples.csv (sample-only subset)
                                  │
                                  ▼
                       06_stage2.ipynb → ... → 08_stage4
```

**What's new vs Stage 1A**

Stage 1A's alias resolution picks **one column** per canonical name and
copies the whole thing. Stage 1B's coalescing picks **one value per
row** by walking a precedence list — and records which source won, so
the lineage is auditable per measurement.

This is the SQL `COALESCE(...)` / PySpark `coalesce(...)` pattern. The
new bit is the row-level source-tracking; see
`05_stage1b.README.md` §7 for the design rationale.


## Parameters

Single tagged `parameters` cell. The default `INPUT_CSV` and `OUT_DIR`
are intentionally `None`. This prevents accidental runs against old local
files. In normal use, `run_pipeline.sh` or Papermill supplies these paths.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Required. The runner passes:
#   INPUT_CSV = <stage1a_out>/data/staged.csv
#   OUT_DIR   = <stage1b_out>
#
# Keep these as None so an interactive run fails clearly instead of
# accidentally using an old local path.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep-merges onto the union of oa_pipeline.schema.DEFAULT_CONFIG and
# oa_pipeline.stage1b.STAGE1B_DEFAULTS. Useful to change precedence lists
# or the safe-for-analysis policy without editing code.
CONFIG_PATH = None

# --- Stage 1B behaviour --------------------------------------------
# If True, skip Parquet writes (CSV only).
NO_PARQUET = False

# Useful during development: prepare everything but write nothing.
DRY_RUN = False

# If True, print preview tables in console form instead of display().
PRINT_COLUMNS = False


## Setup

The best-source coalescing and all the Stage 1B-specific helpers live in
`oa_stage1b.py`. The schema config and value normalisers come from
`oa_schema.py`. `RangePolicy` is a single import from `oa_policy.py` —
no per-notebook redefinition (this was a silent-overwrite bug in the
original monolithic notebook; see README §6).


In [ ]:
from __future__ import annotations

import copy
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import importlib.metadata as importlib_metadata
except Exception:  # pragma: no cover - older Python fallback
    importlib_metadata = None

try:
    from IPython.display import display
except Exception:
    display = None

from oa_pipeline.common import (
    build_flag_summary,
    coerce_datetime,
    coerce_numeric,
    deep_update,
    die,
    ensure_dir,
    make_missingness_table,
    md_table_from_df,
    utc_stamp,
    value_counts_frame,
    write_csv_and_parquet,
    write_json,
    write_text,
)
from oa_pipeline.policy import policy_from_config
from oa_pipeline.schema import DEFAULT_CONFIG, load_config
from oa_pipeline.stage1b import (
    STAGE1B_DEFAULTS,
    add_analysis_range_flags,
    add_best_analysis_fields,
    add_presence_flags,
    add_provenance_fields,
    add_scale_flags,
    add_status_normalizations,
    analysis_ready_subset,
    build_numeric_candidates_for_stage1b,
    provenance_counts_table,
    validate_ta_units,
)


def as_bool(value) -> bool:
    """Parse Papermill friendly boolean values safely."""
    if isinstance(value, bool):
        return value

    if value is None:
        return False

    text = str(value).strip().lower()

    if text in {"true", "1", "yes", "y", "on"}:
        return True

    if text in {"false", "0", "no", "n", "none", "null", "", "off"}:
        return False

    die(f"Cannot parse boolean parameter value: {value!r}")


def normalize_optional_path(value):
    """Return None for blank/None/null-like values, else stripped text."""
    if value is None:
        return None

    text = str(value).strip()
    if text.lower() in {"", "none", "null"}:
        return None

    return text


try:
    oa_pipeline_version = (
        importlib_metadata.version("oa-pipeline")
        if importlib_metadata is not None
        else None
    )
except Exception:
    oa_pipeline_version = None


## Load input CSV and merge config

In [ ]:
# Normalise Papermill parameters before using them.
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)
PRINT_COLUMNS = as_bool(PRINT_COLUMNS)

input_csv_text = normalize_optional_path(INPUT_CSV)
out_dir_text = normalize_optional_path(OUT_DIR)
config_path = normalize_optional_path(CONFIG_PATH)

if input_csv_text is None:
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")

if out_dir_text is None:
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")

src_csv = Path(input_csv_text).expanduser().resolve()
if not src_csv.exists():
    die(
        f"File not found: {src_csv}\n"
        "Did Notebook 04 (Stage 1A) run successfully? Stage 1B reads its staged.csv."
    )

if src_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {src_csv.name}")

out_root = ensure_dir(Path(out_dir_text).expanduser().resolve())

# Merge: schema defaults <- stage1b defaults <- user CONFIG_PATH override.
# Use a deep copy because deep_update may mutate its first input.
config_base = deep_update(copy.deepcopy(DEFAULT_CONFIG), STAGE1B_DEFAULTS)
config, config_source = load_config(config_path)

if config_path:
    # `load_config` merges the user override onto DEFAULT_CONFIG.
    # Re-merge so STAGE1B_DEFAULTS stay above schema defaults, then layer
    # user-provided non-default keys.
    config = deep_update(
        config_base,
        {
            k: v
            for k, v in config.items()
            if k not in DEFAULT_CONFIG or v != DEFAULT_CONFIG.get(k)
        },
    )
else:
    config = config_base
    config_source = None

range_policy = policy_from_config(config)

df = pd.read_csv(src_csv)
if df.empty:
    die(f"Input CSV is empty: {src_csv}")

df["source_file_stage1b"] = str(src_csv)
df["stage1b_processed_utc"] = utc_stamp()

# Cast sample_date and every numeric candidate up front.
if "sample_date" in df.columns:
    coerce_datetime(df, "sample_date")
coerce_numeric(df, build_numeric_candidates_for_stage1b(config))

print(f"Source CSV    : {src_csv}")
print(f"Rows          : {len(df):,}")
print(f"Columns       : {df.shape[1]}")
print(f"Config source : {config_source if config_source else 'DEFAULT_CONFIG + STAGE1B_DEFAULTS'}")
print(f"NO_PARQUET    : {NO_PARQUET}")
print(f"DRY_RUN       : {DRY_RUN}")


## Build best analysis fields

For each variable family, walk the precedence list (configurable) and
pick the first non-NA value per row. Two columns are written per family:

- the value (`ta_best_umolkg`, `ph_best`, `ph_co2sys`, `pco2_best_uatm`,
  `dic_best_umol_kg`)
- the source (`ta_best_source`, `ph_best_source`, ...) — the canonical
  column name the value came from, per row

The pattern is SQL `COALESCE(...)`; the addition is the per-row source.


In [ ]:
meta = add_best_analysis_fields(df, config)

print("TA precedence used :", meta["ta_precedence_used"])
print("pH precedence used :", meta["ph_precedence_used"])
print("ph_co2sys source   :", meta["ph_co2sys_source"])
print("pCO2 precedence    :", meta["pco2_precedence_used"])
print("DIC precedence     :", meta["dic_precedence_used"])


## Normalise QC status, add presence / range / scale flags

Six in-place operations, in order:

1. `add_status_normalizations` — for each QC-status column, write
   trimmed string + uppercased `_norm` version (for case-insensitive
   FAIL/PASS comparisons).
2. `add_provenance_fields` — TA units / pH-scale coalescing + normalisation,
   carbonate-solver defaults, per-variable role flags, and
   `is_sample_row` classification.
3. `validate_ta_units` — flag missing or unexpected TA units.
4. `add_scale_flags` — pH-scale missing for observed and calculated.
5. `add_presence_flags` — `flag_core_chemistry_missing`,
   `flag_pressure_output_dbar_missing` (sample rows only).
6. `add_analysis_range_flags` — range flags on the **best** fields
   (`ta_best_umolkg`, `ph_best`, `ph_co2sys`), not the original
   Stage 1A `ta_umol_kg`/`ph_observed`.


In [ ]:
status_meta     = add_status_normalizations(df, config)
provenance_meta = add_provenance_fields(df, config)
unit_meta       = validate_ta_units(df)
add_scale_flags(df)
add_presence_flags(df)
add_analysis_range_flags(df, policy=range_policy)

print(f"Sample rows (is_sample_row=True): {int((df['is_sample_row'] == True).sum())}")


## Analysis-ready subset

The sample-only subset with two QC gates:

- `safe_for_analysis_qc` — passes if neither `ta_qc_status` nor
  `ph_qc_status` is FAIL. Optionally also blocks rows whose pH came
  from a pH-std-corrected source and whose pH-standard QC failed (set
  by the `phstd_fail_blocks_corrected_ph` policy).
- `safe_for_analysis_strict` — `safe_for_analysis_qc` **AND** no
  range / unit / scale / core-chemistry-missing flag is True.

Two diagnostic columns are always exposed regardless of policy:
`phstd_fail_diagnostic` and `ph_best_from_corrected`.


In [ ]:
missing_tbl = make_missingness_table(df)
df_ready = analysis_ready_subset(df, config)

n_qc = int(df_ready["safe_for_analysis_qc"].sum())
n_strict = int(df_ready["safe_for_analysis_strict"].sum())
print(f"Analysis-ready (samples)    : {len(df_ready):,}")
print(f"  safe_for_analysis_qc      : {n_qc:,}")
print(f"  safe_for_analysis_strict  : {n_strict:,}")


## Quick previews (interactive run only)

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "station_id", "depth_m",
        "ta_best_umolkg", "ta_best_source",
        "ph_best", "ph_best_source",
        "ph_co2sys",
        "pco2_best_uatm", "dic_best_umol_kg",
        "ta_units_normalized",
        "ph_scale_observed_normalized",
        "pressure_output_dbar",
        "safe_for_analysis_qc", "safe_for_analysis_strict",
    ]
    if c in df.columns or c in df_ready.columns
]

if PRINT_COLUMNS:
    print(missing_tbl.head(20).to_string(index=False))
    print("\n=== Analysis fields preview ===")
    print(df[[c for c in preview_cols if c in df.columns]].head(10).to_string(index=False))
    print("\n=== Analysis-ready preview ===")
    print(df_ready[[c for c in preview_cols if c in df_ready.columns]].head(10).to_string(index=False))
elif display is not None:
    display(missing_tbl.head(15))
    display(df[[c for c in preview_cols if c in df.columns]].head(10))
    display(df_ready[[c for c in preview_cols if c in df_ready.columns]].head(10))


## Prepare output paths

Layout — same JWST-style "identity in folder, role in filename" used by
Notebooks 02 / 04. No workbook stem, no stage tag accumulation, no
nested `<stem>/` folder. Pick `OUT_DIR` per run if you need to keep
multiple variants side by side.

```
<OUT_DIR>/
    data/
        analysis_fields.csv             (and .parquet)   # all rows + best fields
        analysis_ready_samples.csv      (and .parquet)   # ◄── Stage 2 input
    reports/
        report.md
    logs/
        manifest.json
        effective_config.json
        missingness.csv
```


In [ ]:
data_dir    = ensure_dir(out_root / "data")
reports_dir = ensure_dir(out_root / "reports")
logs_dir    = ensure_dir(out_root / "logs")

paths = {
    "analysis_fields_csv":          data_dir    / "analysis_fields.csv",
    "analysis_fields_parquet":      data_dir    / "analysis_fields.parquet",
    "analysis_ready_samples_csv":   data_dir    / "analysis_ready_samples.csv",
    "analysis_ready_samples_parquet": data_dir  / "analysis_ready_samples.parquet",
    "report_md":                    reports_dir / "report.md",
    "manifest_json":                logs_dir    / "manifest.json",
    "effective_config_json":        logs_dir    / "effective_config.json",
    "missingness_csv":              logs_dir    / "missingness.csv",
}
print(f"Output root: {out_root}")


## Write outputs

In [ ]:
parquet_written = {"analysis_fields": False, "analysis_ready_samples": False}
parquet_errors  = {"analysis_fields": None,  "analysis_ready_samples": None}

if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    if NO_PARQUET:
        df.to_csv(paths["analysis_fields_csv"], index=False)
        df_ready.to_csv(paths["analysis_ready_samples_csv"], index=False)
        parquet_errors["analysis_fields"] = "Parquet disabled by user"
        parquet_errors["analysis_ready_samples"] = "Parquet disabled by user"
    else:
        ok_a, err_a = write_csv_and_parquet(
            df, paths["analysis_fields_csv"], paths["analysis_fields_parquet"]
        )
        parquet_written["analysis_fields"], parquet_errors["analysis_fields"] = ok_a, err_a
        ok_r, err_r = write_csv_and_parquet(
            df_ready, paths["analysis_ready_samples_csv"], paths["analysis_ready_samples_parquet"]
        )
        parquet_written["analysis_ready_samples"], parquet_errors["analysis_ready_samples"] = ok_r, err_r

    missing_tbl.to_csv(paths["missingness_csv"], index=False)
    write_json(paths["effective_config_json"], config)

    # Build the markdown report
    flag_summary = build_flag_summary(df)
    prov_df = provenance_counts_table(df)

    qc_blocks = []
    for col, title in [
        ("ta_qc_status", "TA QC status distribution"),
        ("ph_qc_status", "Observed pH QC status distribution"),
        ("phstd_status", "pH standard QC status distribution"),
    ]:
        if col in df.columns:
            qc_blocks.append(f"### {title}\n{md_table_from_df(value_counts_frame(df, col))}")
    qc_text = "\n\n".join(qc_blocks) if qc_blocks else "_(No QC status columns found.)_"

    source_blocks = []
    for col, title in [
        ("ta_best_source", "Best TA source distribution"),
        ("ph_best_source", "Best observed pH source distribution"),
        ("pco2_best_source", "Best pCO2 source distribution"),
        ("dic_best_source", "Best DIC source distribution"),
    ]:
        if col in df.columns:
            source_blocks.append(f"### {title}\n{md_table_from_df(value_counts_frame(df, col))}")
    source_text = "\n\n".join(source_blocks) if source_blocks else "_(No source columns found.)_"

    ap = config.get("analysis_policy", {})
    phstd_note = (
        "FAILED pH standards block corrected-observed-pH rows in safe_for_analysis_qc."
        if ap.get("phstd_fail_blocks_corrected_ph") else
        "FAILED pH standards are kept as a diagnostic; do not block by default."
    )
    pressure_note = (
        "Pressure is required for safe_for_analysis_strict."
        if ap.get("require_pressure_for_strict") else
        "Pressure remains a provenance / warning field by default."
    )

    report_md_text = f"""# Stage 1B Report

**Generated:** {utc_stamp()}
**Source CSV:** `{src_csv}`
**Rows:** {len(df):,}
**Columns:** {df.shape[1]:,}

## What Stage 1B does
1. Loads Stage 1A's staged.csv
2. Builds best-source fields for TA, observed pH, calculated pH, pCO2, DIC
3. Normalises QC status, TA units, pH scales
4. Adds chemistry provenance + completeness flags
5. Produces the sample-only analysis-ready subset

## pH policy
- `ph_best` precedence used: `{meta.get("ph_precedence_used", [])}`
- `ph_co2sys` is kept SEPARATE from `ph_best` (observed measurement
  chain vs CO2SYS-derived; downstream code can compare).
- `phstd_status` policy: **{phstd_note}**

## TA policy
- `ta_best_umolkg` precedence used: `{meta.get("ta_precedence_used", [])}`
- TA unit handling reads from the normalised `ta_units_normalized`.

## Chemistry provenance summary
- Carbonate solver: **{provenance_meta.get("carbonate_solver")}**
- Carbon input pair: **{provenance_meta.get("carbon_input_pair_used")}**
- Pressure rule: **{pressure_note}**

{md_table_from_df(prov_df, max_rows=50) if not prov_df.empty else "_(No provenance summary available.)_"}

## Source tracking
{source_text}

## Missingness inventory (top 50)
{md_table_from_df(missing_tbl.head(50), max_rows=200)}

## Range policy used
- TA: {range_policy.ta_min} to {range_policy.ta_max} umol/kg
- pH: {range_policy.ph_min} to {range_policy.ph_max}
- Salinity: {range_policy.sal_min} to {range_policy.sal_max}
- Depth: {range_policy.depth_min} to {range_policy.depth_max} m

## Flag summary
{md_table_from_df(flag_summary, max_rows=200) if not flag_summary.empty else "_(No flags were added.)_"}

## QC rollups
{qc_text}

## Analysis subset summary
- Sample rows: **{int((df['is_sample_row'] == True).sum())}**
- Analysis-ready rows: **{len(df_ready):,}**
- Rows safe_for_analysis_qc: **{int(df_ready['safe_for_analysis_qc'].sum()) if 'safe_for_analysis_qc' in df_ready.columns else 0}**
- Rows safe_for_analysis_strict: **{int(df_ready['safe_for_analysis_strict'].sum()) if 'safe_for_analysis_strict' in df_ready.columns else 0}**

## Main outputs
- analysis_fields CSV         : `{paths["analysis_fields_csv"]}`
- analysis_ready_samples CSV  : `{paths["analysis_ready_samples_csv"]}`  (Stage 2 reads this)
"""
    write_text(paths["report_md"], report_md_text)

    # Manifest
    manifest = {
        "notebook": "05_stage1b",
        "generated_utc": utc_stamp(),
        "input_csv": str(src_csv),
        "output_root": str(out_root),
        "config_source": config_source,
        "parameters": {
            "INPUT_CSV": str(src_csv),
            "OUT_DIR": str(out_root),
            "CONFIG_PATH": CONFIG_PATH,
            "config_path_resolved": config_path,
            "NO_PARQUET": NO_PARQUET,
            "DRY_RUN": DRY_RUN,
            "PRINT_COLUMNS": PRINT_COLUMNS,
        },
        "policy": {
            "range_policy": asdict(range_policy),
            "ta_precedence": config.get("ta_precedence", []),
            "ph_precedence": config.get("ph_precedence", []),
            "ph_co2sys_candidates": config.get("ph_co2sys_candidates", []),
            "pco2_precedence": config.get("pco2_precedence", []),
            "dic_precedence": config.get("dic_precedence", []),
            "analysis_policy": config.get("analysis_policy", {}),
            "provenance_defaults": config.get("provenance_defaults", {}),
            "variable_roles": config.get("variable_roles", {}),
        },
        "field_sources": {**meta, **status_meta, **unit_meta, **provenance_meta},
        "row_counts": {
            "analysis_fields_rows": int(len(df)),
            "analysis_ready_rows": int(len(df_ready)),
            "sample_rows": int((df["is_sample_row"] == True).sum()) if "is_sample_row" in df.columns else None,
            "rows_missing_ta_units": int((df["flag_ta_units_missing"] == True).sum()) if "flag_ta_units_missing" in df.columns else None,
            "rows_unexpected_ta_units": int((df["flag_ta_units_unexpected"] == True).sum()) if "flag_ta_units_unexpected" in df.columns else None,
            "rows_missing_observed_ph_scale": int((df["flag_ph_scale_observed_missing"] == True).sum()) if "flag_ph_scale_observed_missing" in df.columns else None,
            "rows_missing_pressure_output_dbar": int((df["flag_pressure_output_dbar_missing"] == True).sum()) if "flag_pressure_output_dbar_missing" in df.columns else None,
            "rows_safe_for_analysis_qc": int(df_ready["safe_for_analysis_qc"].sum()) if "safe_for_analysis_qc" in df_ready.columns else None,
            "rows_safe_for_analysis_strict": int(df_ready["safe_for_analysis_strict"].sum()) if "safe_for_analysis_strict" in df_ready.columns else None,
        },
        "parquet_written": parquet_written,
        "parquet_errors": parquet_errors,
        "outputs": {k: str(v) for k, v in paths.items()},
        "package_versions": {
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "oa_pipeline": oa_pipeline_version,
        },
    }
    write_json(paths["manifest_json"], manifest)

    print("\nStage 1B complete.")
    print(f"  -> Stage 2 input: {paths['analysis_ready_samples_csv']}")


## Review outputs

In [ ]:
if not DRY_RUN:
    outputs_df = pd.DataFrame(
        {"output_name": list(paths.keys()), "path": [str(p) for p in paths.values()]}
    )
    if display is not None:
        display(outputs_df)
        print("\nAnalysis-fields preview:")
        display(df.head(10))
        print("\nAnalysis-ready (samples) preview:")
        display(df_ready.head(10))
    else:
        print(outputs_df.to_string(index=False))
